# Download Gaia DR4 epoch data and available DataLink products

Read the complete microlensing catalogue and selected-source rows from
`gaia_source`, `all_source_astrometry` and `all_source_photometry`. Retrieve all
available DataLink products for the selected sources as complete FITS ZIP bundles.
Retain quality/rejection flags and image HDUs, validate returned files, and save
query ledgers, checksums and a file/HDU inventory.

The default requests one source. Change `DATALINK_SOURCE_IDS` for a different
source or set it to `[]` and `DATALINK_SOURCE_LIMIT = None` for the entire catalogue.
The final reading/export cell writes per-source epoch FITS files and provenance
to `data/<source_id>/`, ready for notebook 07. Products vary by source.


The data are described in the Gaia DR4 publication by **Wyrzykowski et al. (2026)**.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from io import BytesIO
from getpass import getpass
import gzip, hashlib, json, os, re, shutil, socket, subprocess, tempfile, time, zipfile
import xml.etree.ElementTree as ET

import numpy as np
from astropy.io import fits
from astropy.table import Table
from astroquery.gaia import GaiaClass
from IPython.display import display

ARCHIVE_URL = "https://gea.esac.esa.int/"
TAP_SCHEMA = "gaiadr4"
DATA_RELEASE = "Gaia DR4"
DATA_STRUCTURE = "DATAMODEL_STANDARD"
AUTHENTICATE = False  # Public access; enable only if this endpoint requires login

GAIA_USERNAME = "YOUR_GAIA_USERNAME"
AUTH_METHOD = "prompt"  # "prompt" works on all platforms; "keychain" on macOS
KEYCHAIN_SERVICE = "gaia-archive"
SMOKE_SOURCE_ID = "704967037090946688"


def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "01_archive_access.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the gaia-dr4-microlensing-notebooks folder.")

ROOT = find_project_root(Path.cwd())

OUTPUT = ROOT / "outputs" / TAP_SCHEMA / "microlensing_datalink_example"
DATALINK_SOURCE_IDS = ["4048864055145726976"]  # Same example source as notebook 07
DATALINK_SOURCE_LIMIT = 1  # None = all microlensing sources
BATCH_SIZE = 10  # A conservative example batch, not a documented service maximum
JOB_WAIT_SECONDS = 1800
NETWORK_TIMEOUT_SECONDS = 120  # Socket inactivity timeout, restored after the run

OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG = dict(endpoint=ARCHIVE_URL, schema=TAP_SCHEMA, data_release=DATA_RELEASE,
              data_structure=DATA_STRUCTURE, retrieval_type="ALL", format="fits", valid_data=False)
print(CONFIG)  # No credentials are included in this dictionary.

## Archive helpers and optional authentication

Public access is the default (`AUTHENTICATE = False`). If access requires an
archive account, enable authentication, enter your username, and use the private
password prompt. The optional macOS Keychain mode uses your own `gaia-archive`
entry. Credentials are never written to a notebook or output file, and an
authenticated session always logs out in `finally`.

Each session begins with a fresh known-source TAP query. Requests are sequential;
authenticated mode checks the shared account limits of five executing and 100
queued jobs. ADQL, job IDs, dates, original VOTables and endpoint/schema/release
settings are retained. Timed-out submitted jobs are resumed, and TAP errors or
overflow are rejected. Keep source IDs as exact integers or decimal strings.


Cached DataLink ZIPs are reused after validation. Failed requests, valid empty
responses and absent products are recorded separately. Run only one downloader
per output directory. No automatic retries or parallel submissions are made.

In [ ]:
def now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2))

def source_id(value):
    if isinstance(value, (float, np.floating)):
        raise ValueError("A floating-point source ID may already have lost precision.")
    value = str(value)
    if not re.fullmatch(r"[0-9]{1,19}", value) or int(value) <= 0:
        raise ValueError("Source IDs must be positive decimal strings or exact integers.")
    return value

def identifier(value):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", value):
        raise ValueError("Invalid ADQL identifier")
    return value

def authenticate(gaia):
    if not GAIA_USERNAME.strip() or GAIA_USERNAME == "YOUR_GAIA_USERNAME":
        raise ValueError("Set GAIA_USERNAME to your own authorised archive account.")
    captured = None
    password = None
    try:
        if AUTH_METHOD == "keychain":
            captured = subprocess.run(
                ["security", "find-generic-password", "-a", GAIA_USERNAME,
                 "-s", KEYCHAIN_SERVICE, "-w"],
                capture_output=True, text=True, check=True, timeout=15)
            password = captured.stdout.rstrip("\n")
        elif AUTH_METHOD == "prompt":
            password = getpass("Gaia archive password: ")
        else:
            raise ValueError("AUTH_METHOD must be prompt or keychain")
        if not password:
            raise ValueError("Empty password")
        gaia.login(user=GAIA_USERNAME, password=password, verbose=False)
    finally:
        del password, captured
        # Astroquery also retains passwords internally; discard those references.
        for client in (gaia, getattr(gaia, "_GaiaClass__gaiadata", None)):
            if client is not None and hasattr(client, "_TapPlus__pwd"):
                client._TapPlus__pwd = None

def checked_votable(path):
    payload = path.read_bytes()
    if payload[:2] == b"\x1f\x8b":
        payload = gzip.decompress(payload)
    root = ET.fromstring(payload)
    statuses = [n.get("value", "").upper() for n in root.iter()
                if n.tag.split("}")[-1] == "INFO" and n.get("name", "").upper() == "QUERY_STATUS"]
    if not statuses or any(s != "OK" for s in statuses):
        raise RuntimeError(f"TAP output is incomplete or failed: QUERY_STATUS={statuses}")
    result = Table.read(BytesIO(payload), format="votable")
    # Normalize archive column names without changing values or units.
    for column in list(result.colnames):
        if column.lower() != column:
            result.rename_column(column, column.lower())
    return result

def tap(gaia, query, name, refresh=False):
    signature = json.dumps(dict(CONFIG, query=query), sort_keys=True)
    key = name + "_" + hashlib.sha256(signature.encode()).hexdigest()[:12]
    folder = OUTPUT / "tap" / key
    folder.mkdir(parents=True, exist_ok=True)
    result_file, ledger = folder / "result.vot", folder / "job.json"
    (folder / "query.adql").write_text(query)
    state = json.loads(ledger.read_text()) if ledger.exists() else dict(CONFIG, query=query, created=now())
    if state.get("status") == "complete":
        if result_file.exists() and not refresh:
            return checked_votable(result_file)
        if refresh:
            # Refresh only a completed query. A submitted/timed-out job is resumed.
            state = dict(CONFIG, query=query, created=now())
    if state.get("job_id"):
        job = gaia.load_async_job(jobid=state["job_id"], load_results=False, verbose=False)
    else:
        jobs = gaia.list_async_jobs(verbose=False) if AUTHENTICATE else []
        phases = [j.get_phase().upper() for j in jobs]
        if phases.count("EXECUTING") >= 5 or phases.count("QUEUED") >= 100:
            raise RuntimeError("Shared account job capacity is full; inspect existing jobs before retrying.")
        job = gaia.launch_job_async(query, name=name, output_file=str(result_file),
            output_format="votable", dump_to_file=True, background=True, verbose=False)
        state.update(job_id=str(job.jobid), status="submitted")
        save_json(ledger, state)  # Persist immediately, before polling.
    deadline = time.monotonic() + JOB_WAIT_SECONDS
    while True:
        phase = job.get_phase(update=True).upper()
        state.update(phase=phase, checked=now())
        save_json(ledger, state)
        if phase == "COMPLETED":
            break
        if phase in {"ERROR", "ABORTED"}:
            raise RuntimeError(f"TAP job {job.jobid} is {phase}; inspect it before replacement.")
        if time.monotonic() > deadline:
            raise TimeoutError(f"Job {job.jobid} retained for resumption; no replacement was submitted.")
        time.sleep(2)
    job.outputFile = str(result_file)
    job.outputFileUser = str(result_file)
    job.save_results(verbose=False)  # Preserve the original server response.
    result = checked_votable(result_file)
    result.write(folder / "result.ecsv", format="ascii.ecsv", overwrite=True)
    state.update(status="complete", rows=len(result), completed=now(), query_status_checked=True)
    save_json(ledger, state)
    return result

## Complete DataLink bundles and a FITS inventory

`retrieval_type="ALL"` asks for all products available for those source IDs;
availability varies. This can include epoch photometry and astrometry, epoch RV
parameters, spectra, posterior samples and image products. The inventory records
what actually came back. An unreturned product is not assumed to exist.

Each request uses `valid_data=False`: rejected observations and their flags are
retained. Apply explicit scientific quality cuts later. Cached bundles are reused
only after ZIP CRC and FITS validation. Empty successful responses, failed requests
and unattempted batches remain distinguishable in the ledgers.

Some Astroquery versions download the complete FITS ZIP and then fail while
trying to parse image HDUs as tables. Here a **newly returned, independently
validated ZIP** is still usable: the parsing exception type is recorded, and
tables/images are read directly with Astropy. No request is changed to a retired
DataLink structure name. A failed request without a valid ZIP raises an error.

In [ ]:
@contextmanager
def working_directory(folder):
    previous = Path.cwd()
    os.chdir(folder)
    try:
        yield
    finally:
        os.chdir(previous)

def inventory_bundle(bundle):
    rows = []
    with zipfile.ZipFile(bundle) as archive:
        bad = archive.testzip()
        if bad:
            raise RuntimeError(f"Bad ZIP CRC: {bad}")
        for member in archive.infolist():
            if member.is_dir():
                continue
            if not member.filename.lower().endswith((".fits", ".fit", ".fits.gz")):
                rows.append(dict(member=member.filename, hdu=-1, kind="auxiliary", name="", rows=0,
                                 source_ids="", bytes=member.file_size))
                continue
            with fits.open(BytesIO(archive.read(member)), memmap=False, checksum=True) as hdus:
                hdus.verify("exception")
                for index, hdu in enumerate(hdus):
                    sid = str(hdu.header.get("SOURCEID", ""))
                    if not sid:
                        found = re.findall(r"(?<![0-9])([0-9]{16,19})(?![0-9])", member.filename)
                        sid = ";".join(found)
                    is_table = isinstance(hdu, (fits.BinTableHDU, fits.TableHDU))
                    rows.append(dict(member=member.filename, hdu=index, kind="table" if is_table else "image/header",
                        name=hdu.name, rows=int(hdu.header.get("NAXIS2", 0)) if is_table else 0,
                        source_ids=sid, bytes=member.file_size))
    return rows

def datalink(gaia, ids):
    request = dict(CONFIG, source_ids=ids)
    key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()[:16]
    folder = OUTPUT / "datalink" / key
    folder.mkdir(parents=True, exist_ok=True)
    bundle, ledger = folder / "products.zip", folder / "request.json"
    state = dict(request, started=now(), status="started")
    if bundle.exists():
        state = json.loads(ledger.read_text()) if ledger.exists() else state
    else:
        save_json(ledger, state)
        try:
            with tempfile.TemporaryDirectory(dir=folder) as scratch:
                client_error = None
                with working_directory(scratch):
                    try:
                        gaia.load_data(ids=ids, data_release=DATA_RELEASE,
                            data_structure=DATA_STRUCTURE, retrieval_type="ALL", format="fits",
                            valid_data=False, dump_to_file=True, overwrite_output_file=False, verbose=False)
                    except Exception as error:
                        client_error = type(error).__name__
                returned = list(Path(scratch).glob("datalink_output_*.zip"))
                if len(returned) != 1:
                    raise RuntimeError("No unique complete DataLink ZIP returned; not evidence of absent data.")
                inventory_bundle(returned[0])  # Validate before accepting even after a client parsing error.
                shutil.move(str(returned[0]), bundle)
                state["client_exception_type"] = client_error
        except Exception as error:
            state.update(status="failed", exception_type=type(error).__name__, checked=now())
            save_json(ledger, state)
            raise
    rows = inventory_bundle(bundle)
    for row in rows:
        row["bundle"] = str(bundle.relative_to(OUTPUT))
    returned_ids = set(s for row in rows for s in row["source_ids"].split(";") if s in ids)
    state.update(status="complete" if rows else "empty_response", checked=now(), hdu_inventory_rows=len(rows),
                 source_ids_without_identified_products=sorted(set(ids) - returned_ids),
                 sha256=hashlib.sha256(bundle.read_bytes()).hexdigest())
    save_json(ledger, state)
    save_json(folder / "inventory.json", rows)
    if state.get("client_exception_type"):
        print("Downloaded ZIP validated directly; client parsing raised", state["client_exception_type"])
    return rows

## Run: smoke test → metadata → catalogue → selected-source rows → DataLink

Edit the configuration first, then run this cell. Metadata is inspected before
constructing the catalogue projections. The complete microlensing table is read
directly, so missing consolidated `gaia_source` rows cannot remove events through
an inner join. The three source tables are retrieved separately for selected IDs.
Missing table rows and missing products are recorded separately from failures.

No automatic retries or parallel jobs are submitted. On a service/quota error,
rerun later: successful cached queries and ZIPs are reused and saved TAP jobs are
resumed. If you interrupt execution, `finally` normally closes the session; the
optional final logout cell is also available.

In [ ]:
def run_example():
    identifier(TAP_SCHEMA)
    if BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be positive")
    if DATALINK_SOURCE_LIMIT is not None and DATALINK_SOURCE_LIMIT < 1:
        raise ValueError("Use a positive source limit or None")
    global gaia  # Allows the optional cleanup cell after a manual interruption.
    previous_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(NETWORK_TIMEOUT_SECONDS)
    gaia = None
    try:
        gaia = GaiaClass(gaia_tap_server=ARCHIVE_URL, gaia_data_server=ARCHIVE_URL,
                         show_server_messages=False)
        gaia.ROW_LIMIT = -1
        if AUTHENTICATE:
            authenticate(gaia)
        smoke = tap(gaia, f"SELECT source_id FROM {TAP_SCHEMA}.gaia_source WHERE source_id={source_id(SMOKE_SOURCE_ID)}", "smoke", refresh=True)
        if len(smoke) != 1:
            raise RuntimeError("Known-source TAP test did not return one row; check access/schema/source ID.")
        names = ["vari_microlensing", "gaia_source", "all_source_astrometry", "all_source_photometry"]
        # Inspect table metadata before constructing projections.
        # This avoids non-ASCII text in binary TAP_SCHEMA metadata responses.
        columns, metadata = {}, []
        for name in names:
            table = gaia.load_table(f"{TAP_SCHEMA}.{name}", verbose=False)
            columns[name] = [str(c.name).lower() for c in table.columns]
            metadata.append(dict(table=f"{TAP_SCHEMA}.{name}", columns=columns[name]))
        save_json(OUTPUT / "columns.json", dict(CONFIG, inspected=now(), tables=metadata))
        for name in names:
            if "source_id" not in columns[name]:
                raise RuntimeError(f"Metadata does not advertise {name}.source_id; check the current schema.")
            for column in columns[name]:
                identifier(column)
        projection = ",".join(columns['vari_microlensing'])
        counts = tap(gaia, f"SELECT COUNT(*) AS n FROM {TAP_SCHEMA}.vari_microlensing", "microlensing_count")
        events = tap(gaia, f"SELECT {projection} FROM {TAP_SCHEMA}.vari_microlensing ORDER BY source_id", "microlensing_all_columns")
        if len(events) != int(counts['n'][0]):
            raise RuntimeError("Microlensing row count disagrees with COUNT(*); catalogue is not validated complete.")
        ids = sorted(set(source_id(v) for v in events['source_id']), key=int)
        selected = list(dict.fromkeys(source_id(v) for v in DATALINK_SOURCE_IDS)) if DATALINK_SOURCE_IDS else ids
        if DATALINK_SOURCE_LIMIT is not None:
            selected = selected[:DATALINK_SOURCE_LIMIT]
        save_json(OUTPUT / "selection.json", dict(CONFIG, retrieved=now(), catalogue_rows=len(events),
            catalogue_unique_sources=len(ids), selected_source_ids=selected))
        events.write(OUTPUT / "microlensing_all_columns.ecsv", format="ascii.ecsv", overwrite=True)
        print(f"Validated {len(events)} microlensing rows; selected {len(selected)} sources for DataLink.")
        # Separate bounded source-table queries retain all advertised columns.
        for start in range(0, len(selected), BATCH_SIZE):
            batch = selected[start:start+BATCH_SIZE]
            for name in names[1:]:
                projection = ",".join(columns[name])
                tap(gaia, f"SELECT {projection} FROM {TAP_SCHEMA}.{name} WHERE source_id IN ({','.join(batch)}) ORDER BY source_id",
                    f"{name}_batch_{start//BATCH_SIZE+1:04d}")
        inventory = []
        for start in range(0, len(selected), BATCH_SIZE):
            inventory.extend(datalink(gaia, selected[start:start+BATCH_SIZE]))
        save_json(OUTPUT / "datalink_inventory.json", inventory)
        print(f"Validated {len(inventory)} returned file/HDU inventory entries. Products vary by source.")
        return events, inventory
    finally:
        try:
            if gaia is not None and AUTHENTICATE:
                gaia.logout(verbose=False)
        finally:
            socket.setdefaulttimeout(previous_timeout)

events, inventory = run_example()
display(events[:5])
if inventory:
    display(Table(rows=inventory)[:20])

## Read returned tables and images

The JSON inventory identifies every retained bundle, file and FITS HDU. The
example below opens one epoch-photometry or astrometry file when available;
change the search text to select spectra, RV parameters, flags or image products.
`Table.read(hdu)` reads a table with FITS metadata; image arrays are available as
`hdus[index].data`. Keep the original bundle as the authoritative raw download.

FITS headers and the inventory retain source/release provenance. Arrays may hold
multiple CCD observations in a single row; a table row is not necessarily an
accepted or independent measurement. Do not filter away flags before inspection.

In [ ]:
def open_product(entry):
    bundle = OUTPUT / entry['bundle']
    with zipfile.ZipFile(bundle) as archive:
        return fits.open(BytesIO(archive.read(entry['member'])), memmap=False)

examples = [r for r in inventory if r['kind'] == 'table' and
            any(tag in r['member'].upper() for tag in ('EPOCH_PHOTOMETRY', 'EPOCH_ASTROMETRY'))]
if examples:
    chosen = examples[0]
    with open_product(chosen) as hdus:
        hdus.info()
        data = Table.read(hdus[chosen['hdu']])
        display(data[:5])
else:
    print("No such epoch table was returned for this selection; inspect datalink_inventory.json for other products.")

# To inspect an image, choose an inventory entry with kind == 'image/header',
# open it using open_product(entry), and inspect hdus[entry['hdu']].data.


def export_epoch_inputs(inventory):
    """Copy unchanged per-source FITS members and record their download provenance."""
    selection = json.loads((OUTPUT / 'selection.json').read_text())
    for sid in selection['selected_source_ids']:
        folder = ROOT / 'data' / sid
        ledger = folder / 'provenance.json'
        previous = json.loads(ledger.read_text()) if ledger.exists() else None
        if previous and any(previous.get(k) != v for k, v in CONFIG.items()):
            raise RuntimeError(f'Existing epoch inputs have different release/endpoint settings: {folder}')
        files, missing = {}, []
        for product in ('EPOCH_PHOTOMETRY', 'EPOCH_ASTROMETRY'):
            candidates = {(r['bundle'], r['member']) for r in inventory
                          if r['kind'] == 'table' and sid in r['source_ids'].split(';')
                          and re.match(product + r'[-_ ]GAIA(?:[-_ ]|$)', Path(r['member']).name.upper())}
            if not candidates:
                missing.append(product)
                continue
            if len(candidates) != 1:
                raise RuntimeError(f'Ambiguous {product} files for source {sid}; inspect the inventory.')
            bundle, member = candidates.pop()
            with zipfile.ZipFile(OUTPUT / bundle) as archive:
                payload = archive.read(member)
            target = folder / (product.lower() + '.fits')
            checksum = hashlib.sha256(payload).hexdigest()
            if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != checksum:
                raise FileExistsError(f'Epoch input already exists with a different checksum: {target}')
            folder.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
            files[target.name] = dict(product=product, sha256=checksum,
                                      bundle=str(OUTPUT / bundle), member=member)
        folder.mkdir(parents=True, exist_ok=True)
        save_json(ledger, dict(CONFIG, source_id=sid, exported_utc=now(),
                              selection_record=str(OUTPUT / 'selection.json'),
                              files=files, products_not_returned=missing))
        print(f'Exported {len(files)} epoch FITS files for {sid}; products not returned: {missing}')

export_epoch_inputs(inventory)

## Cleanup and reproducibility

Authenticated sessions log out in `finally`; public access uses no password.
Restart the kernel and clear outputs before sharing an executed notebook.
Endpoint, TAP schema and DataLink release must be checked independently when
archive services change. Keep raw ZIP bundles and the download provenance.

In [ ]:
# Optional manual cleanup:
# gaia.logout(verbose=False)